In [0]:
from pyspark.sql import functions as F

base_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/"

df = spark.read.format("delta").load(
    f"{base_path}Silver/flight_airport_delta"
)

df = df.dropDuplicates()

df = df.dropna(subset=["FlightDate", "Origin", "Dest"])

df = df.withColumn("Origin", F.upper(F.trim(F.col("Origin"))))
df = df.withColumn("Dest", F.upper(F.trim(F.col("Dest"))))

df.write.format("delta").mode("overwrite").save(
    f"{base_path}Silver/flight_cleaned"
)

clean_df = spark.read.format("delta").load(
    f"{base_path}Silver/flight_cleaned"
)

display(clean_df)

In [0]:
from pyspark.sql import functions as F

df = spark.read \
    .option("header", True) \
    .csv("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing /On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_4.csv")

df = df.withColumns({
    "schedule_crsdeparture_minutes":
        F.floor(F.col("CRSDepTime").cast("double") / 100) * 60 + F.col("CRSDepTime").cast("double") % 100,

    "schedule_departure_minutes":
        F.floor(F.col("DepTime").cast("double") / 100) * 60 + F.col("DepTime").cast("double") % 100,

    "schedule_crsarrival_minutes":
        F.floor(F.col("CRSArrTime").cast("double") / 100) * 60 + F.col("CRSArrTime").cast("double") % 100,

    "schedule_arrival_minutes":
        F.floor(F.col("ArrTime").cast("double") / 100) * 60 + F.col("ArrTime").cast("double") % 100
})
display(df)

In [0]:
from pyspark.sql import functions as F

df = df.withColumns({
    "sched_dep_min": F.floor(F.col("CRSDepTime") / 100) * 60 + F.col("CRSDepTime") % 100,
    "sched_arr_min": F.floor(F.col("CRSArrTime") / 100) * 60 + F.col("CRSArrTime") % 100
})

In [0]:
df = df.withColumns({
    "is_cancelled": F.col("Cancelled").cast("double") == 1,
    "is_diverted": F.col("Diverted").cast("double") == 1,
    "is_dep_delayed": F.col("DepDelay").cast("double") > 0,
    "is_arr_delayed": F.col("ArrDelay").cast("double") > 0
})
display(df)

In [0]:
df = df.withColumns({
    "flight_quarter": F.col("Quarter"),
    "day_of_week": F.col("DayOfWeek"),
    "is_weekend": F.col("DayOfWeek").isin(6, 7)
})

display(df)

In [0]:
df = df.withColumn(
    "arr_delay_catogary",
    F.when(F.col("ArrDelay").cast("double") < 0, "early")
     .when(F.col("ArrDelay").cast("double").between(0, 15), "on time")
     .when(F.col("ArrDelay").cast("double").between(16, 45), "minor delay")
     .when(F.col("ArrDelay").cast("double").between(40, 120), "major delay")
     .otherwise("unknown delay")
)


display(df)

In [0]:
# new column sum_delay_cause - carrier delay , weather delay and NSA delay and secure delay and late aircraft delay add all the value
df_sum = df.withColumn(
    "sum_delay_cause",
    F.coalesce(F.col("CarrierDelay").cast("double"), F.lit(0)) +
    F.coalesce(F.col("WeatherDelay").cast("double"), F.lit(0)) +
    F.coalesce(F.col("NASDelay").cast("double"), F.lit(0)) +
    F.coalesce(F.col("SecurityDelay").cast("double"), F.lit(0)) +
    F.coalesce(F.col("LateAircraftDelay").cast("double"), F.lit(0))
)

display(df_sum)

In [0]:
# new column based on condition data_quality_delay_dispatchrd - arrival delay should not be NULL , flight delay is cancelled or is_cancelled check
df_condition = df_sum.withColumn(
    "data_quality_delay_dispatchrd",
    F.when(F.col("ArrDelay").isNotNull() & (F.col("Cancelled").cast("double") == 0), 1).otherwise(0)
)
df_abs = df_condition.withColumn(
    "absolute_delay",
    F.abs(F.col("ArrDelay").cast("double") - F.col("sum_delay_cause").cast("double"))
)

df_cancel = df_abs.withColumn(
    "cancelation_category",
    F.when(F.col("CancellationCode") == "A", "carrier")
     .when(F.col("CancellationCode") == "B", "weather")
     .when(F.col("CancellationCode") == "C", "NAS")
     .when(F.col("CancellationCode") == "D", "security")
     .otherwise("unknown")
)

display(df_cancel)


In [0]:
# diffference between arrival delay and sum_delay_cau -> so create a new column absolute delay
df_delay = df_cancel.withColumn(
    "absolute_delat",
    F.abs(F.col("ArrDelay").cast("double") - F.col("sum_delay_cause").cast("double"))
)
display(df_delay)

In [0]:
# new column cancelation_category condition canceled value A (carrier ) , b(weather) , C (air_delay) , D (security)
df_1 = df.withColumn(
    "cancelation_category",
    F.when(F.col("CancellationCode") == "A", "carrier")
     .when(F.col("CancellationCode") == "B", "weather")
     .when(F.col("CancellationCode") == "C", "NAS")
     .when(F.col("CancellationCode") == "D", "security")
     .otherwise("unknown")
)
display(df_1)

In [0]:
from pyspark.sql import functions as F

# Read airport JSON
airport = spark.read \
    .option("multiline", True) \
    .json("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing /airports.json")

# Drop existing geo columns from previous runs to avoid duplicates after joins
df = df.drop("origin_latitude", "origin_longitude", "dest_latitude", "dest_longitude")

# Create origin lookup
origin = airport.select(
    F.col("iata").alias("origin_iata"),
    F.col("latitude").alias("origin_latitude"),
    F.col("longitude").alias("origin_longitude")
)

# Origin join
df = df.join(
    F.broadcast(origin),
    F.col("Origin") == F.col("origin_iata"),
    "left"
).drop("origin_iata")

# Create destination lookup
dest = airport.select(
    F.col("iata").alias("dest_iata"),
    F.col("latitude").alias("dest_latitude"),
    F.col("longitude").alias("dest_longitude")
)

# Destination join
df = df.join(
    F.broadcast(dest),
    F.col("Dest") == F.col("dest_iata"),
    "left"
).drop("dest_iata")

display(df)

In [0]:
df_airport_dest = spark.read \
    .option("multiline", True) \
    .json("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing /airports.json") \
    .select(
        F.col("iata").alias("dest_iata"),
        F.col("latitude").alias("dest_latitude"),
        F.col("longitude").alias("dest_longitude")
    )

df = df.join(
    F.broadcast(df_airport_dest),
    df["Dest"] == df_airport_dest["dest_iata"],
    "left"
).drop("dest_iata")

display(df)

In [0]:
df = df.withColumns({
    "origin_lat": F.col("origin_latitude").cast("double"),
    "origin_lon": F.col("origin_longitude").cast("double"),
    "dest_lat": F.col("dest_latitude").cast("double"),
    "dest_lon": F.col("dest_longitude").cast("double")
})

In [0]:
R = 6371

df = df.withColumn(
    "haversine_distance",
    2 * R * F.asin(
        F.sqrt(
            F.sin(F.radians(F.col("dest_lat") - F.col("origin_lat")) / 2) ** 2
            +
            F.cos(F.radians(F.col("origin_lat"))) *
            F.cos(F.radians(F.col("dest_lat"))) *
            F.sin(F.radians(F.col("dest_lon") - F.col("origin_lon")) / 2) ** 2
        )
    )
)

In [0]:
from pyspark.sql import functions as F

df = spark.read \
    .option("header", True) \
    .csv("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing /On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_4.csv")

airport = spark.read \
    .option("multiline", True) \
    .json("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Landing /airports.json")

# Origin
origin = airport.select(
    F.col("iata").alias("origin_iata"),
    F.col("latitude").alias("origin_latitude"),
    F.col("longitude").alias("origin_longitude")
)

df = df.join(
    F.broadcast(origin),
    df["Origin"] == origin["origin_iata"],
    "left"
).drop("origin_iata")

# Destination
dest = airport.select(
    F.col("iata").alias("dest_iata"),
    F.col("latitude").alias("dest_latitude"),
    F.col("longitude").alias("dest_longitude")
)

df = df.join(
    F.broadcast(dest),
    df["Dest"] == dest["dest_iata"],
    "left"
).drop("dest_iata")

display(df)

In [0]:
df = df.withColumns({
    "origin_lat": F.col("origin_latitude").cast("double"),
    "origin_lon": F.col("origin_longitude").cast("double"),
    "dest_lat": F.col("dest_latitude").cast("double"),
    "dest_lon": F.col("dest_longitude").cast("double")
})

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/flight_final_delta")

display(
    spark.read
    .format("delta")
    .load("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/flight_final_delta")
)